# 🚀 7강. Streamlit 앱 고급 + ResNet-18 도입 (AI Pair) 💡

> 📌 **이 노트북의 목표**
> - RandomForest 앱(6강)을 "1위 장르만 보여주는 앱"에서 **"얼마나 확신하는지까지 보여주는 앱"**으로 업그레이드합니다.
> - Streamlit의 기억 장치 `st.session_state`로 예측 이력을 쌓고, 여러 곡을 한 화면에서 비교합니다.
> - ImageNet으로 이미 학습된 ResNet-18의 마지막 레이어만 바꿔 장르 분류기로 재활용하는 **전이학습**의 개념을 노트북에서 먼저 체험합니다(실제 학습은 8강).

> 📍 **앞 노트북(5강·6강)** — 5강에서 RandomForest로 장르를 분류하고, 6강에서 그 모델을 `joblib`으로 저장해 `app.py`를 만들었습니다. 오늘은 그 위에 신뢰도·이력·멀티파일·ResNet 개념을 쌓습니다.

---

## 이전 챕터와 연결 (5강 → 6강 → 7강)

| 5강 | 6강 | 7강 |
|---|---|---|
| RF 모델 학습 | `app.py` 기본 완성 | `app_v2.py` — 신뢰도 + 이력 + 멀티파일 |
| 멜스펙트로그램 이해 | `st.file_uploader` | ResNet-18 개념 도입 |

---

## 학습 목표 3가지

| # | 목표 | 핵심 도구 |
|---|------|----------|
| 1 | 예측 확률 분포를 bar chart로 시각화하고 `st.session_state`로 이력을 관리할 수 있다 | `st.session_state`, `st.bar_chart` |
| 2 | 여러 곡을 동시에 비교하는 Streamlit 멀티파일 앱을 구현할 수 있다 | `accept_multiple_files=True` |
| 3 | ImageNet 전이학습의 직관과 ResNet-18의 마지막 레이어 교체 코드를 설명할 수 있다 | `torchvision.models.resnet18` |

---

## 목차

- [환경 설정](#env)
- [섹션 1 — 신뢰도 bar chart + 확률 분포 이해](#sec1)
- [섹션 2 — st.session_state 이력 관리](#sec2)
- [섹션 3 — 멀티파일 비교 업로드](#sec3)
- [섹션 4 — 전이학습 개념 도입 + ResNet-18](#sec4)
- [섹션 5 — app_v2.py 생성 + 실행 안내](#sec5)
- [마무리 — AI Pair 섹션 + 8강 예고](#fin)

> 🧭 **이 강이 다루지 않는 것**
> - ResNet-18을 실제로 **학습(fine-tuning)**시키는 것 → 데이터 로더·학습 루프·손실 곡선은 **8강**에서 다룹니다. 오늘은 "구조를 이해하고 가중치를 안 바꾼 채 추론만" 해봅니다.
> - MFCC·멜스펙트로그램을 처음부터 추출하는 원리 → 4강·5강에서 이미 다뤘습니다. 오늘은 저장된 모델·피처를 **재사용**합니다.
> - Streamlit Cloud 실제 배포(도메인 연결, secrets 관리) → 6강에서 다룬 로컬 실행 안내 수준까지만 다룹니다.

> 📋 **오늘의 계약(contract)** — 이 셋이 맞으면 절반은 성공입니다
> 1. **입력**: WAV 파일 경로 1개 (`predict_with_confidence`는 내부에서 3초만 잘라 사용)
> 2. **출력**: `(top_genre: str, prob_dict: dict[str, float])` — `prob_dict`는 10개 장르 키를 모두 가지며 확률 내림차순 정렬
> 3. **검증 한 줄**: `abs(sum(prob_dict.values()) - 1.0) < 1e-5` — softmax/RF 확률은 항상 합이 1에 수렴합니다

> 🎯 **오늘 강의 트랙 안내** — ★ 기본 vs ★★ 심화
>
> | 트랙 | 대상 | 무엇을 |
> |---|---|---|
> | ★ 기본 트랙 | 전원 필수 (CPU만 있어도 완주) | 신뢰도 bar chart · `st.session_state` 이력 · 멀티파일 비교 · ResNet-18 구조 이해(가중치는 바꾸지 않음) |
> | ★★ 심화 트랙 | GPU/MPS 보유 학생 | ResNet-18로 실제 PNG 스펙트로그램 추론 실행, `freeze_backbone` 옵션 비교 |
>
> 잠시 뒤 계산되는 `RUN_HEAVY` 값이 이 트랙을 가릅니다 — GPU/MPS가 없으면 자동으로 `False`가 되어 ★★ 항목은 시뮬레이션 결과로 대체됩니다(GPU 없이도 무엇이 나오는지는 확인 가능합니다). 아래에서 ★★ 표시가 붙은 셀들은 이 노트북 끝 **AI Pair 섹션 · Solo 레벨 2(심화 트랙)** 과제로 그대로 이어지니, 지나가면서 눈여겨봐 두세요.

In [1]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 1/16] 패키지 설치                                  │
# │ 입력: 없음   출력: 설치 로그 (이미 있으면 skip)        │
# └────────────────────────────────────────────────────────┘
import subprocess, sys

packages = [
    'librosa',
    'scikit-learn',
    'joblib',
    'streamlit',
    'torch',
    'torchvision',
    'seaborn',
]

for pkg in packages:
    import_name = pkg.replace('-', '_')
    try:
        __import__(import_name)
        print(f'[OK] {pkg} 이미 설치됨')
    except ImportError:
        print(f'[설치 중] {pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print(f'[완료] {pkg}')

print('\n모든 패키지 준비 완료!')

[OK] librosa 이미 설치됨
[설치 중] scikit-learn ...



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


[완료] scikit-learn
[OK] joblib 이미 설치됨


[OK] streamlit 이미 설치됨


[OK] torch 이미 설치됨


[OK] torchvision 이미 설치됨


[OK] seaborn 이미 설치됨

모든 패키지 준비 완료!


In [2]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 2/16] 전역 초기화 — import + 한글폰트 + 경로      │
# │ 입력: 없음   출력: 환경 정보                           │
# └────────────────────────────────────────────────────────┘
import os, sys, platform, random, warnings, pathlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
# ── librosa 가드 (NumPy 2.4 / numba lazy-import 충돌 대비) — 이후 배포 단계(Streamlit Cloud)와 동일한 lazy-import 가드 패턴 ──
# [왜] librosa import는 성공해도 librosa.load() 호출 시 numba가 lazy import되며
#      NumPy 2.4 환경에서 ImportError 발생 → probe로 미리 감지
try:
    import librosa
    import librosa.display
    import io as _io
    import numpy as _np_probe
    # 0.1초짜리 무음 WAV를 메모리에서 생성해 librosa.load probe
    import soundfile as _sf
    _buf = _io.BytesIO()
    _sf.write(_buf, _np_probe.zeros(2205, dtype='float32'), 22050, format='WAV')
    _buf.seek(0)
    librosa.load(_buf, sr=22050)
    LIBROSA_AVAILABLE = True
    print('[librosa] 정상 로드 (probe 통과)')
except (ImportError, Exception) as e:
    librosa = None
    LIBROSA_AVAILABLE = False
    print(f'[librosa] 사용 불가 ({type(e).__name__}: {e})')
    print('  → scipy.signal 폴백 모드 활성화 (NumPy 2.4 호환)')
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pathlib import Path

warnings.filterwarnings('ignore')

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[librosa] 정상 로드 (probe 통과)


In [3]:
# [셀 2 계속] 재현성 시드 고정 + 한글 폰트 준비 import

# ── 재현성 시드 고정 ──────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── 한글 폰트 설정 (macOS/Windows/Linux robust fallback) ──────────
import matplotlib
import matplotlib.font_manager as fm
from pathlib import Path
import platform

In [4]:
# [셀 2 계속] 한글 폰트 후보 목록 (OS별) — setup_korean_font()가 참조하는 룩업 테이블
# [왜] 함수 본문에 두면 함수 정의 셀만 45줄을 넘기므로, 데이터(룩업 테이블)와 로직(함수)을 분리합니다
FONT_PREFERRED_BY_SYSTEM = {
    "Darwin": [
        "AppleGothic", "Apple SD Gothic Neo", "NanumGothic",
        "Noto Sans CJK KR", "Noto Sans KR",
    ],
    "Windows": [
        "Malgun Gothic", "맑은 고딕", "NanumGothic",
        "Noto Sans CJK KR", "Noto Sans KR",
    ],
    "Linux": [
        "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR",
        "Noto Sans CJK JP", "Noto Sans CJK SC",
    ],
}
FONT_COMMON_FILES = {
    "Darwin": [
        "/System/Library/Fonts/AppleGothic.ttf",
        "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
        "/Library/Fonts/NanumGothic.ttf",
        str(Path.home() / "Library/Fonts/NanumGothic.ttf"),
    ],
    "Windows": [
        "C:/Windows/Fonts/malgun.ttf",
        "C:/Windows/Fonts/NanumGothic.ttf",
    ],
    "Linux": [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansKR-Regular.otf",
    ],
}

In [5]:
# [셀 2 계속] 한글 폰트 설정 함수 정의 — 위 룩업 테이블에서 사용 가능한 폰트를 고른다
def setup_korean_font():
    system = platform.system()
    for font_path in FONT_COMMON_FILES.get(system, []):
        if Path(font_path).exists():
            fm.fontManager.addfont(font_path)

    available = {f.name for f in fm.fontManager.ttflist}
    candidates = FONT_PREFERRED_BY_SYSTEM.get(system, []) + [
        "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR",
        "AppleGothic", "Malgun Gothic",
    ]
    chosen = next((font for font in candidates if font in available), None)

    # Linux(Colab)엔 한글 폰트가 기본 설치돼 있지 않을 수 있어, 없으면 apt-get으로
    # 나눔고딕을 설치한 뒤 다시 탐색합니다.
    if chosen is None and system == "Linux":
        import os
        os.system("apt-get -qq -y install fonts-nanum > /dev/null 2>&1")
        for _fp in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
            fm.fontManager.addfont(_fp)
        available = {f.name for f in fm.fontManager.ttflist}
        chosen = next((font for font in candidates if font in available), None)

    if chosen is None:
        chosen = "DejaVu Sans"
        print("[폰트 경고] 한글 폰트를 찾지 못했습니다. 그래프 한글이 깨지면 NanumGothic 또는 Noto Sans CJK KR을 설치하세요.")

    matplotlib.rcParams["font.family"] = chosen
    matplotlib.rcParams["font.sans-serif"] = [
        chosen, "NanumGothic", "Noto Sans CJK KR", "AppleGothic", "Malgun Gothic", "DejaVu Sans",
    ]
    matplotlib.rcParams["axes.unicode_minus"] = False
    try:
        import seaborn as sns
        sns.set_theme(
            style="whitegrid",
            rc={
                "font.family": chosen,
                "font.sans-serif": [
                    chosen, "NanumGothic", "Noto Sans CJK KR", "AppleGothic", "Malgun Gothic", "DejaVu Sans",
                ],
                "axes.unicode_minus": False,
            },
        )
    except Exception:
        pass
    return chosen

In [6]:
# [셀 2 계속] 폰트 적용 + 디바이스 자동 감지 + RUN_HEAVY 게이트 + 데이터 경로 설정

KOREAN_FONT = setup_korean_font()
print(f"[폰트] {KOREAN_FONT}")


# ── 디바이스 자동 감지 (MPS → CUDA → CPU) ────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif platform.system() == 'Darwin' and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'[디바이스] {DEVICE}')

# ── RUN_HEAVY 게이트 (★★ GPU 트랙) ───────────────────────
RUN_HEAVY = (DEVICE.type in ('cuda', 'mps'))
print(f'[RUN_HEAVY] {RUN_HEAVY}  → ★★ ResNet-18 실행: {"예" if RUN_HEAVY else "아니오 (시뮬레이션)"}')

# ── 데이터 경로 설정 ──────────────────────────────────────
# cwd와 상위 폴더 중 실제로 Data/가 있는 쪽을 NB_DIR로 씁니다(Data/·모델 파일은 AI_Music/ 루트에 있습니다).
_cwd, _parent = Path('.').resolve(), Path('..').resolve()
NB_DIR = _cwd if (_cwd / 'Data').exists() else _parent
DATA_ROOT = NB_DIR / 'Data/Music_genres'
CSV_PATH  = DATA_ROOT / 'features_3_sec.csv'
WAV_ROOT  = DATA_ROOT / 'genres_original'
PNG_ROOT  = DATA_ROOT / 'images_original'
MODEL_PATH = NB_DIR / 'model_rf.joblib'
LE_PATH    = NB_DIR / 'label_encoder.joblib'
SC_PATH    = NB_DIR / 'scaler.joblib'

print(f'[경로] DATA_ROOT={DATA_ROOT.resolve()}')
print(f'  CSV={CSV_PATH.exists()}, WAV={WAV_ROOT.exists()}, PNG={PNG_ROOT.exists()}')
print(f'  모델={MODEL_PATH.exists()}, LE={LE_PATH.exists()}, SC={SC_PATH.exists()}')

[폰트] AppleGothic
[디바이스] mps
[RUN_HEAVY] True  → ★★ ResNet-18 실행: 예
[경로] DATA_ROOT=/Users/sanguinekim/Documents/[준비중] KDT_AI Human 강의/Part II. Transformer/music_ai_v2/Data/Music_genres
  CSV=True, WAV=True, PNG=True
  모델=True, LE=True, SC=True


In [7]:
# [셀 2 계속] 색상 팔레트 + 장르 목록 + FEATURE_COLS

# ── 색상 팔레트 (STYLE_GUIDE 표준) ───────────────────────
COLORS = {
    'primary':   '#2563EB',
    'secondary': '#7C3AED',
    'accent':    '#059669',
    'neutral':   '#6B7280',
    'warning':   '#F59E0B',
}

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

# ── FEATURE_COLS (6강과 동일 순서) ─────────────────────────
FEATURE_COLS = (
    ['chroma_stft_mean','chroma_stft_var',
     'rms_mean','rms_var',
     'spectral_centroid_mean','spectral_centroid_var',
     'spectral_bandwidth_mean','spectral_bandwidth_var',
     'rolloff_mean','rolloff_var',
     'zero_crossing_rate_mean','zero_crossing_rate_var',
     'harmony_mean','harmony_var',
     'perceptr_mean','perceptr_var',
     'tempo']
    + [f'mfcc{i}_{s}' for i in range(1, 21) for s in ('mean', 'var')]
)

print('\n초기화 완료!')


초기화 완료!


```
# ┌────────────────────────────────────────────────────────┐
# │ 오늘 배울 것 3가지                                     │
# │  1. 신뢰도 bar chart — 점 예측이 아닌 확률 분포로      │
# │  2. session_state 이력 — 여러 곡 결과를 테이블로       │
# │  3. ResNet-18 전이학습 — 이미지 분류기를 소리에 재활용  │
# └────────────────────────────────────────────────────────┘
```

---
<a id='sec1'></a>
## 섹션 1 — 6강 복습 + 신뢰도 bar chart

### 왜 '확률 분포'가 필요한가?

> 6강의 `app.py`는 1위 장르만 표시했습니다.  
> 예측이 `jazz 87%`, `blues 8%`인 곡과  
> `jazz 35%`, `blues 32%`인 곡은 **같은 '1위: jazz'이지만 완전히 다른 신뢰도**입니다.  
> 점 예측(point prediction) 대신 **확률 분포(probability distribution)**를 보여줘야  
> 사용자가 모델을 올바르게 신뢰할 수 있습니다.

6강에서 배운 `predict_proba()`가 반환하는 10개 확률 값이 이 bar chart의 원천입니다.

> 💡 [3강 회상] 3강에서 했던 KMeans 클러스터링이 받은 **오디오 피처 벡터**와, 
> 5강 RandomForest가 받은 벡터는 동일한 형식(features_3_sec.csv의 한 행)입니다.
> 즉 비지도(3강) → 지도(5강) → 더 강력한 모델(7강~8강)로 모델 진화를 보고 있습니다.

> ▶ **실행 전 예측** — jazz 곡 하나를 넣기 전에 먼저 적어보세요.
> - 이 곡의 예측 확률이 **한 장르에 90% 이상 몰릴까요**, 아니면 **여러 장르에 고르게 퍼질까요**?
> - 1위와 2위 확률의 차이가 크면(예: 90% vs 3%) 무엇을 뜻할까요? 반대로 작으면(예: 35% vs 32%)?
>
> 💡 정답은 숫자가 아니라 **경향**입니다 — 장르마다 "얼마나 다른 장르와 소리가 겹치는가"가 다르기 때문입니다.

### 📐 코드를 읽기 전에 — `predict_with_confidence`가 하는 일

| 코드 | 하는 일 | 비유 |
|---|---|---|
| `extract_features(wav_path)` | WAV → 57개 숫자 벡터 | 곡의 "건강검진 수치표"를 뽑는 일 |
| `sc.transform(vec)` | 각 피처를 학습 때와 같은 스케일로 정규화 | 검진 수치를 병원마다 다른 단위 대신 같은 기준으로 환산 |
| `rf.predict_proba(vec_sc)` | 10개 장르 각각의 확률 반환 | 심사위원 100명(트리 100그루)의 투표 비율 — "몇 명이 jazz에 손을 들었나" |
| 확률 내림차순 정렬 | 1위부터 보기 좋게 정리 | 투표 결과 순위표 작성 |

In [8]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 3/16] 6강 모델 로드 (없으면 즉석 학습) — 함수 정의 │
# └────────────────────────────────────────────────────────┘
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
def load_or_train_rf():
    """6강에서 저장된 모델 로드, 없으면 즉석 학습."""
    if MODEL_PATH.exists() and LE_PATH.exists() and SC_PATH.exists():
        rf = joblib.load(MODEL_PATH)
        le = joblib.load(LE_PATH)
        sc = joblib.load(SC_PATH)
        print(f'[로드 완료] 6강 모델 파일 3개 로드')
        return rf, le, sc

    print('[안내] 6강 모델 파일 없음 → 즉석 학습 시작')
    if CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH)
        X  = df[FEATURE_COLS].values.astype(np.float32)
        y_raw = df['label'].values
    else:
        print('[폴백] CSV 없음 → 합성 데이터 1000샘플')
        n = 1000
        X = np.random.randn(n, len(FEATURE_COLS)).astype(np.float32)
        y_raw = np.array(GENRES * (n // len(GENRES) + 1))[:n]

    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_sc, y_tr)
    acc = accuracy_score(y_te, rf.predict(X_te_sc))
    print(f'  학습 완료 — Test Accuracy: {acc:.4f}')

    joblib.dump(rf, MODEL_PATH)
    joblib.dump(le, LE_PATH)
    joblib.dump(sc, SC_PATH)
    print(f'  모델 저장 완료')
    return rf, le, sc

In [9]:
# [셀 3 계속] 함수 실행 — 6강 모델 로드 또는 즉석 학습
rf, le, sc = load_or_train_rf()
print(f'\n  RF 클래스: {le.classes_}')

[로드 완료] 6강 모델 파일 3개 로드

  RF 클래스: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']


In [10]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 4/16] extract_features 재정의 (6강와 동일)          │
# │ 입력: WAV 경로   출력: np.ndarray (1, 57)              │
# └────────────────────────────────────────────────────────┘
# [왜] 이 함수는 6강와 100% 동일 — 학습 때와 같은 피처 순서를 보장해야 함

def extract_features(wav_path: str) -> np.ndarray:
    if not LIBROSA_AVAILABLE:
        raise RuntimeError(
            'librosa 미설치 환경에서 extract_features() 호출 불가.\n'
            'RUN_HEAVY=False 환경에서는 features_3_sec.csv를 직접 사용하세요.'
        )
    y_audio, sr = librosa.load(str(wav_path), sr=22050, mono=True, duration=3.0)
    feats = {}
    chroma = librosa.feature.chroma_stft(y=y_audio, sr=sr)
    feats['chroma_stft_mean'] = float(np.mean(chroma))
    feats['chroma_stft_var']  = float(np.var(chroma))
    rms_f = librosa.feature.rms(y=y_audio)
    feats['rms_mean'] = float(np.mean(rms_f))
    feats['rms_var']  = float(np.var(rms_f))
    sc_f = librosa.feature.spectral_centroid(y=y_audio, sr=sr)
    feats['spectral_centroid_mean'] = float(np.mean(sc_f))
    feats['spectral_centroid_var']  = float(np.var(sc_f))
    bw = librosa.feature.spectral_bandwidth(y=y_audio, sr=sr)
    feats['spectral_bandwidth_mean'] = float(np.mean(bw))
    feats['spectral_bandwidth_var']  = float(np.var(bw))
    ro = librosa.feature.spectral_rolloff(y=y_audio, sr=sr)
    feats['rolloff_mean'] = float(np.mean(ro))
    feats['rolloff_var']  = float(np.var(ro))
    zcr = librosa.feature.zero_crossing_rate(y_audio)
    feats['zero_crossing_rate_mean'] = float(np.mean(zcr))
    feats['zero_crossing_rate_var']  = float(np.var(zcr))
    harm, perc = librosa.effects.hpss(y_audio)
    feats['harmony_mean']  = float(np.mean(harm))
    feats['harmony_var']   = float(np.var(harm))
    feats['perceptr_mean'] = float(np.mean(perc))
    feats['perceptr_var']  = float(np.var(perc))
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    feats['tempo'] = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])
    mfcc_f = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
    for i in range(20):
        feats[f'mfcc{i+1}_mean'] = float(np.mean(mfcc_f[i]))
        feats[f'mfcc{i+1}_var']  = float(np.var(mfcc_f[i]))
    vec = np.array([feats[col] for col in FEATURE_COLS], dtype=np.float32)
    return vec.reshape(1, -1)

In [11]:
# [셀 4 계속] 정의 확인 출력
print('[함수 정의 완료] extract_features → shape (1, 57)')
print('[확인] FEATURE_COLS 길이:', len(FEATURE_COLS))

[함수 정의 완료] extract_features → shape (1, 57)
[확인] FEATURE_COLS 길이: 57


In [12]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 5/16] predict_with_confidence() — 전체 확률 반환  │
# │ 입력: WAV 경로   출력: (1위장르, {장르: 확률} 딕셔너리)│
# └────────────────────────────────────────────────────────┘
# [왜] 6강의 predict_genre는 top-3만 반환 → 전체 확률 분포를 bar chart로 보여주려면
#      10개 장르 모두의 확률이 필요함

def predict_with_confidence(wav_path: str) -> tuple:
    """
    Returns:
        top_genre  (str): 1위 장르
        prob_dict  (dict): {장르: 확률} 10개 전체 — bar chart용
    """
    vec    = extract_features(wav_path)          # (1, 57)
    vec_sc = sc.transform(vec)                   # 스케일 적용
    proba  = rf.predict_proba(vec_sc)[0]         # (10,) 확률 배열

    # 장르 이름 ↔ 확률 매핑
    prob_dict = {le.classes_[i]: float(proba[i]) for i in range(len(proba))}

    # 확률 내림차순 정렬
    prob_dict = dict(sorted(prob_dict.items(), key=lambda x: x[1], reverse=True))
    top_genre = list(prob_dict.keys())[0]

    return top_genre, prob_dict

In [13]:
# [셀 5 계속] 테스트 (LIBROSA_AVAILABLE 가드)
if not LIBROSA_AVAILABLE:
    print('[건너뜀] librosa 미사용 환경 — predict_with_confidence() 테스트 생략')
    print('  features_3_sec.csv 기반 추론은 RF 모델 섹션에서 진행합니다.')
    _genre, _probs = 'N/A', {g: 0.0 for g in GENRES}
elif WAV_ROOT.exists():
    _test_wav = WAV_ROOT / 'jazz' / 'jazz.00000.wav'
    _genre, _probs = predict_with_confidence(_test_wav)
    print(f'1위 장르: {_genre}')
    print('\n전체 확률 분포:')
    for g, p in _probs.items():
        bar = '█' * int(p * 40)
        print(f'  {g:<12} {p:.3f}  {bar}')
else:
    import tempfile, scipy.io.wavfile as _wf
    _sr  = 22050
    _t   = np.linspace(0, 3.0, int(_sr * 3.0), endpoint=False)
    _sig = (np.sin(2 * np.pi * 440 * _t) * 32767).astype(np.int16)
    _tmpf = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    _wf.write(_tmpf.name, _sr, _sig)
    _genre, _probs = predict_with_confidence(_tmpf.name)
    print(f'1위 장르: {_genre}')
    print('\n전체 확률 분포:')
    for g, p in _probs.items():
        bar = '█' * int(p * 40)
        print(f'  {g:<12} {p:.3f}  {bar}')

1위 장르: jazz

전체 확률 분포:
  jazz         0.820  ████████████████████████████████
  classical    0.050  ██
  pop          0.050  ██
  country      0.040  █
  rock         0.030  █
  disco        0.010  
  blues        0.000  
  hiphop       0.000  
  metal        0.000  
  reggae       0.000  


In [14]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 6/16] 신뢰도 bar chart 시각화                      │
# │ 입력: prob_dict   출력: matplotlib 수평 bar chart      │
# └────────────────────────────────────────────────────────┘
# [왜] 수평 바차트: 장르 이름(긴 텍스트)을 y축에 놓으면 가독성 UP
#      1위 장르는 색상 강조, 나머지는 연하게
def plot_confidence_bar(prob_dict: dict, title: str = '') -> plt.Figure:
    """전체 장르 확률 분포 수평 bar chart. Streamlit st.pyplot(fig)에 바로 사용."""
    genres_sorted = list(prob_dict.keys())
    probs_sorted  = list(prob_dict.values())
    top_genre     = genres_sorted[0]

    bar_colors = [
        COLORS['primary'] if g == top_genre else COLORS['neutral']
        for g in genres_sorted
    ]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(
        genres_sorted[::-1],   # 위에서부터 높은 순 → 뒤집기
        probs_sorted[::-1],
        color=bar_colors[::-1],
        edgecolor='white', height=0.6, alpha=0.88
    )

    # 확률 값 레이블
    for bar, prob in zip(bars, probs_sorted[::-1]):
        if prob > 0.01:
            ax.text(
                bar.get_width() + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{prob:.1%}', va='center', fontsize=9,
                color=COLORS['primary'] if prob == max(probs_sorted) else COLORS['neutral']
            )

    ax.set_xlim(0, 1.12)
    ax.set_xlabel('예측 확률')
    ax.set_title(title or f'장르 예측 신뢰도 — 1위: {top_genre.upper()}',
                 fontsize=12, fontweight='bold')
    ax.axvline(0.5, color=COLORS['warning'], lw=1.2, linestyle='--',
               label='50% 기준선')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)
    fig.tight_layout()
    return fig

In [15]:
# [셀 6 계속] 신뢰도 bar chart 호출 예시
fig_conf = plot_confidence_bar(_probs, title=f'신뢰도 bar chart 예시')
plt.show()
print('\n[학생 실험] 확률이 고르게 퍼져 있는 곡과 한 장르에 몰린 곡을 비교해보세요')


[학생 실험] 확률이 고르게 퍼져 있는 곡과 한 장르에 몰린 곡을 비교해보세요


> **🔍 결과 해석**
> - 1위 장르 확률이 50% 미만이면 모델이 자신 없다는 신호 — 여러 장르 특성이 혼재하는 곡
> - 확률이 한 장르에 90% 이상 몰리면 고신뢰 예측 — classical처럼 음색이 독특한 장르에서 자주 발생
> - 실무 시사점: 서비스에서 신뢰도 < 40% 곡은 "분류 불확실" 경고를 표시하는 것이 사용자 경험에 좋음

> 🤖 **AI Agent에서 이렇게 씁니다** — 방금 만든 "1위 장르만 보지 말고 확률 분포 전체를 보여준다"는 원칙은 LLM 에이전트에도 그대로 적용됩니다. 좋은 RAG 에이전트는 검색 결과를 하나의 정답처럼 툭 던지지 않고, 근거 문서가 얼마나 확신을 뒷받침하는지 함께 제시하거나(신뢰도가 낮으면 "확실하지 않습니다"라고 말함), 여러 개의 후보 답변 중 어느 것을 택할지 확률/점수 기반으로 라우팅합니다. 이 강의 뒷부분(Phase 2~3)에서 만들 LangGraph 에이전트의 `add_conditional_edges`도 결국 "이 상황에서 어느 노드로 갈지"를 신뢰도·조건값으로 분기하는 것 — 오늘 만든 신뢰도 bar chart가 그 축소판입니다.

> 🎲 **불확실성 참고** — `load_or_train_rf()`는 `model_rf.joblib`이 이미 있으면 그대로 로드하므로 결과가 항상 동일합니다. 하지만 파일이 없어 즉석 재학습이 실행되는 환경이라면, `random_state=42`로 시드를 고정해도 scikit-learn 버전이나 CPU 스레드 수에 따라 소수점 단위의 정확도 차이가 날 수 있습니다. 이 노트북이 보고하는 정확도·확률 수치는 "seed=42, 로컬 CPU 기준"이며, 절대값보다는 "RF가 어떤 장르에서 자신 있어 하는가"라는 **경향**을 보는 데 집중하세요.

---
<a id='sec2'></a>
## 섹션 2 — st.session_state 이력 관리

### 왜 이력(History)이 필요한가?

> Streamlit은 사용자가 버튼을 누를 때마다 스크립트 전체를 재실행합니다.  
> 이전 파일을 업로드한 결과가 사라집니다 — **기억이 없는 앱**입니다.  
> `st.session_state`는 Streamlit의 "메모리"입니다.  
> 업로드할 때마다 결과를 리스트에 쌓으면 → 테이블로 이력 표시 가능.  

| 개념 | Python 유사 개념 |
|---|---|
| `st.session_state['history']` | 전역 딕셔너리 — 탭을 닫을 때까지 유지 |
| 새 파일 업로드 | 리스트에 append |

In [16]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 7/16] st.session_state 이력 관리 — 노트북 시뮬레이션│
# │ 실제 앱에서는 session_state를 씁니다.                  │
# │ 여기서는 일반 리스트로 동일 동작 시뮬레이션            │
# └────────────────────────────────────────────────────────┘
# [왜] session_state['history']는 Python list와 동일하게 작동
#      여기서 list로 시뮬레이션하면 앱 코드 이해가 쉬워짐

# 이력 초기화 (앱에서는 st.session_state.setdefault('history', []))
prediction_history = []

def record_prediction(filename: str, top_genre: str, prob_dict: dict):
    """예측 결과를 이력에 추가."""
    record = {
        '파일명':   filename,
        '1위 장르': top_genre,
        '1위 확률': f"{prob_dict[top_genre]:.1%}",
        '2위 장르': list(prob_dict.keys())[1],
        '고신뢰도': '✓' if prob_dict[top_genre] >= 0.5 else '△',
    }
    prediction_history.append(record)
    return record

In [17]:
# [셀 7 계속] test_files 구성 — 실제 WAV 우선, 없으면 합성 신호로 폴백
test_files = []
if WAV_ROOT.exists():
    # [왜] jazz·blues를 나란히 넣어야 섹션 3(멀티파일 비교)에서 "blues와 jazz 비교"가
    #      실제로 가능합니다. 마지막 (disco, 00013)은 일부러 저신뢰도 곡을 골랐습니다 —
    #      나머지 6곡은 1위 확률이 66~93%로 전부 "확신" 판정만 나와 대조가 안 됐기 때문입니다.
    for genre, idx in [('jazz', '00000'), ('blues', '00000'), ('classical', '00000'),
                        ('rock', '00000'), ('pop', '00000'), ('metal', '00000'),
                        ('disco', '00013')]:
        p = WAV_ROOT / genre / f'{genre}.{idx}.wav'
        if p.exists():
            test_files.append((genre, p))

if not test_files:
    # 폴백: 합성 신호로 6개 시뮬레이션
    print('[폴백] 실제 WAV 없음 → 합성 신호 6개로 이력 시뮬레이션')
    import tempfile, scipy.io.wavfile as _wf2
    for freq, genre in [(440, 'jazz'), (880, 'pop'), (220, 'blues'), (660, 'rock'), (330, 'metal'), (550, 'disco')]:
        _sr2 = 22050
        _t2  = np.linspace(0, 3.0, int(_sr2 * 3.0), endpoint=False)
        _sig2 = (np.sin(2 * np.pi * freq * _t2) * 32767).astype(np.int16)
        _tmp2 = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        _wf2.write(_tmp2.name, _sr2, _sig2)
        test_files.append((genre, Path(_tmp2.name)))

In [18]:
# [셀 7 계속] 예측 실행 + 이력 테이블 생성
if not LIBROSA_AVAILABLE:
    print('[건너뜀] librosa 미사용 환경 — 이력 시뮬레이션 생략')
    df_hist = pd.DataFrame(columns=['파일명', '1위 장르', '1위 확률', '2위 장르', '고신뢰도'])
else:
    print('이력 시뮬레이션 중...')
    for true_genre, wav_path in test_files:
        top_g, prob_d = predict_with_confidence(wav_path)
        record_prediction(wav_path.name, top_g, prob_d)
        print(f'  {wav_path.name:<25} → 예측={top_g:<12} ({prob_d[top_g]:.1%})')
    df_hist = pd.DataFrame(prediction_history)
    print('\n예측 이력 테이블:')
    print(df_hist.to_string(index=False))

이력 시뮬레이션 중...


  jazz.00000.wav            → 예측=jazz         (82.0%)


  blues.00000.wav           → 예측=blues        (76.0%)


  classical.00000.wav       → 예측=classical    (93.0%)


  rock.00000.wav            → 예측=rock         (66.0%)


  pop.00000.wav             → 예측=pop          (70.0%)


  metal.00000.wav           → 예측=metal        (71.0%)


  disco.00013.wav           → 예측=disco        (41.0%)

예측 이력 테이블:
                파일명     1위 장르 1위 확률     2위 장르 고신뢰도
     jazz.00000.wav      jazz 82.0% classical    ✓
    blues.00000.wav     blues 76.0%      rock    ✓
classical.00000.wav classical 93.0%      jazz    ✓
     rock.00000.wav      rock 66.0%     disco    ✓
      pop.00000.wav       pop 70.0%    reggae    ✓
    metal.00000.wav     metal 71.0%      rock    ✓
    disco.00013.wav     disco 41.0%       pop    △


In [19]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 8/16] 이력 테이블 시각화 — 장르별 고신뢰도 비율   │
# │ 입력: prediction_history   출력: bar chart             │
# └────────────────────────────────────────────────────────┘
# [왜] 이력 데이터를 시각화하면 '어느 장르 곡에서 모델이 자신 없어하는가' 패턴을 파악

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽: 예측된 장르 분포 (파이 차트)
genre_counts = df_hist['1위 장르'].value_counts()
# [왜] test_files 장르 수가 5종을 넘을 수 있음(저신뢰도 disco 포함) ->
#      고정 5색 리스트 대신 COLORS를 순환시켜 색이 겹치지 않게 함
_pie_palette = list(COLORS.values())
pie_colors = [_pie_palette[i % len(_pie_palette)] for i in range(len(genre_counts))]
axes[0].pie(
    genre_counts.values,
    labels=genre_counts.index,
    autopct='%1.0f%%',
    colors=pie_colors,
    startangle=90
)
axes[0].set_title(f'예측 장르 분포 (n={len(df_hist)}곡)', fontsize=11, fontweight='bold')

# 오른쪽: 고신뢰도 비율 막대
high_conf = (df_hist['고신뢰도'] == '✓').sum()
low_conf  = (df_hist['고신뢰도'] == '△').sum()
axes[1].bar(['고신뢰도\n(≥50%)', '저신뢰도\n(<50%)'],
            [high_conf, low_conf],
            color=[COLORS['accent'], COLORS['warning']],
            edgecolor='white', alpha=0.88)
for bar, val in zip(axes[1].patches, [high_conf, low_conf]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        str(val), ha='center', fontsize=11, fontweight='bold'
    )
axes[1].set_title('예측 신뢰도 분포', fontsize=11, fontweight='bold')
axes[1].set_ylabel('곡 수')
sns.despine(ax=axes[1])

plt.suptitle('예측 이력 분석 — st.session_state로 이 그래프를 앱에서 실시간으로',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

> **🔍 결과 해석**
> - 고신뢰도/저신뢰도 비율은 곡의 장르 순도에 비례 — 퓨전 장르는 저신뢰도가 많음
> - 파이 차트: 모델이 특정 장르로 치우쳐 예측하면 학습 데이터 불균형 신호
> - 실무 시사점: 이력 데이터가 쌓이면 오분류 패턴을 찾아 추가 학습 데이터를 수집하는 피드백 루프에 사용
> - 🔭 `df_hist`의 '2위 장르' 컬럼을 눈여겨보세요 — 1위와 2위 확률 차이가 작을수록 그 두 장르가 서로 '헷갈리는 pair'라는 신호입니다. (예: 1위 A장르 45% · 2위 B장르 40%라면 A·B 특징이 비슷해 모델이 확신하지 못하는 경우) 이 아이디어는 AI Pair 섹션 **Solo 레벨 3**에서 직접 확인합니다.

---
<a id='sec3'></a>
## 섹션 3 — 멀티파일 비교 업로드

### 왜 여러 곡을 한 화면에서 비교해야 하는가?

> "이 두 곡 중에 어떤 게 더 jazz스럽지?"  
> 한 번에 여러 파일을 비교하면 **개별 예측보다 훨씬 많은 인사이트**를 얻습니다.  
> Streamlit은 `st.file_uploader(accept_multiple_files=True)` 한 줄로 멀티파일을 지원합니다.  
> 노트북에서 먼저 비교 시각화를 구현하고, 그 코드를 앱에 붙입니다.

In [20]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 9/16] 멀티파일 비교 — grouped bar chart 함수 정의  │
# │ 입력: 여러 WAV 파일   출력: 장르별 그룹 bar chart      │
# └────────────────────────────────────────────────────────┘
def compare_songs_chart(wav_list: list, labels: list) -> plt.Figure:
    """
    여러 곡의 확률 분포를 grouped bar chart로 비교.

    Args:
        wav_list: WAV 파일 경로 리스트
        labels  : 각 파일의 표시 이름 리스트
    Returns:
        matplotlib Figure
    """
    all_probs = []
    for wav_path in wav_list:
        _, prob_d = predict_with_confidence(wav_path)
        # 장르 순서 통일 (GENRES 기준)
        probs_ordered = [prob_d.get(g, 0.0) for g in GENRES]
        all_probs.append(probs_ordered)

    n_files  = len(wav_list)
    n_genres = len(GENRES)
    x        = np.arange(n_genres)
    width    = 0.8 / n_files

    # 곡별 색상
    bar_palette = [COLORS['primary'], COLORS['secondary'],
                   COLORS['accent'], COLORS['warning']]

    fig, ax = plt.subplots(figsize=(13, 5))
    for i, (probs, label) in enumerate(zip(all_probs, labels)):
        offset = (i - n_files / 2 + 0.5) * width
        color  = bar_palette[i % len(bar_palette)]
        ax.bar(x + offset, probs, width,
               label=label, color=color, alpha=0.82, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(GENRES, rotation=30, ha='right')
    ax.set_xlabel('장르')
    ax.set_ylabel('예측 확률')
    ax.set_ylim(0, 1.05)
    ax.set_title('멀티파일 장르 확률 비교 — 같은 축에서 여러 곡을 나란히',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)
    fig.tight_layout()
    return fig

In [21]:
# [셀 9 계속] 비교 실행 (LIBROSA_AVAILABLE 가드)
if not LIBROSA_AVAILABLE:
    print('[건너뜀] librosa 미사용 환경 — 멀티파일 비교 차트 생략')
    print('  librosa 정상 환경에서 실행하면 grouped bar chart가 표시됩니다.')
elif not test_files:
    print('[건너뜀] test_files 없음 — 셀 7/16(이력 시뮬레이션)부터 재실행하세요.')
else:
    compare_wav_list  = [p for _, p in test_files[:3]]
    compare_wav_names = [p.name for p in compare_wav_list]
    fig_compare = compare_songs_chart(compare_wav_list, compare_wav_names)
    plt.show()
    print('\n[학생 실험] blues와 jazz를 동시에 비교해보세요.')


[학생 실험] blues와 jazz를 동시에 비교해보세요.


> 💡 **표준편차(standard deviation)란?** 10개 확률 숫자가 평균에서 서로 얼마나 벌어져 있는지 재는 자입니다. 숫자들이 대부분 비슷하면(예: 전부 0.1 근처) 표준편차가 작고, 어느 하나가 확 튀면(예: 한 곳만 0.9, 나머지는 0에 가까움) 표준편차가 큽니다.

### 🔍 확인해보기 — "고르게 퍼진 확률"을 눈이 아니라 숫자로 확인

앞의 bar chart를 눈으로 보고 "이 곡은 확신도가 낮아 보인다"고 판단했다면, 그 판단을 **표준편차 하나의 숫자**로 검증해봅니다. 확률이 10개 장르에 고르게 퍼져 있으면 표준편차가 작고, 한 장르에 쏠려 있으면 표준편차가 큽니다.

In [22]:
# [확인해보기] 확률 분포의 표준편차 = "확신도 점수"
# 표준편차가 클수록 확률이 한 장르에 쏠려 있다(확신) / 작을수록 고르게 퍼져 있다(불확실)
import numpy as np

if LIBROSA_AVAILABLE and test_files:
    print(f'{"파일":<25} {"1위 장르":<12} {"1위 확률":>8} {"표준편차":>10}  판정')
    for name, wav_path in test_files:
        _, prob_d = predict_with_confidence(wav_path)
        probs_arr = np.array(list(prob_d.values()))
        std = probs_arr.std()
        top_g = list(prob_d.keys())[0]
        judged = '확신' if std > 0.15 else '불확실'
        print(f'{wav_path.name:<25} {top_g:<12} {prob_d[top_g]:>7.1%} {std:>10.4f}  {judged}')
    print()
    print('[검증] bar chart에서 1위 확률이 눈에 띄게 높아 보였던 곡일수록 표준편차도 크게 나오는지 확인하세요.')
else:
    print('[건너뜀] librosa 미사용 환경 — 정량 검증은 librosa 정상 환경에서 실행하세요.')

파일                        1위 장르           1위 확률       표준편차  판정
jazz.00000.wav            jazz           82.0%     0.2408  확신


blues.00000.wav           blues          76.0%     0.2223  확신
classical.00000.wav       classical      93.0%     0.2772  확신


rock.00000.wav            rock           66.0%     0.1926  확신
pop.00000.wav             pop            70.0%     0.2019  확신


metal.00000.wav           metal          71.0%     0.2056  확신
disco.00013.wav           disco          41.0%     0.1192  불확실

[검증] bar chart에서 1위 확률이 눈에 띄게 높아 보였던 곡일수록 표준편차도 크게 나오는지 확인하세요.


> **🔍 멀티파일 비교 결과 해석**
> - 같은 장르 축(예: jazz 열)에서 어느 곡의 막대가 더 높은지 비교하면 → 그 곡이 해당 장르 특징을 더 강하게 가진다는 뜻
> - 한 곡의 막대가 특정 장르 열 하나에서만 압도적으로 높으면 → 장르 색이 뚜렷한 곡, 여러 열에 고르게 퍼지면 → 혼합 장르에 가까운 곡
> - 실무 시사점: 멀티파일 비교를 DJ 믹싱 앱에 활용하면 장르 확률 프로필이 비슷한 곡을 추천할 수 있음

---
<a id='sec4'></a>
## 섹션 4 — 전이학습 개념 도입 + ResNet-18

```
# ┌────────────────────────────────────────────────────────┐
# │ 전이학습(Transfer Learning) 핵심 직관                  │
# │  문제: 장르 PNG 이미지가 ~940장 — CNN 처음부터 학습은 │
# │        데이터 부족                                     │
# │  해결: ImageNet 100만 장으로 학습된 필터를 재사용      │
# │        마지막 FC(=Fully Connected, 완전연결층)만 교체  │
# │        (1000클래스 → 10클래스)                          │
# └────────────────────────────────────────────────────────┘
```

### 왜 전이학습인가? (5강 RF와 비교)

> 5강의 RandomForest는 **사람이 설계한 57개 피처**(MFCC, chroma 등)를 씁니다.  
> 피처 설계가 잘못되면 성능이 제한됩니다.  
> ResNet-18은 **픽셀 자체**를 입력으로 받아 피처를 스스로 학습합니다.  
> 단, 데이터가 적으면 처음부터 학습하기 어려우므로 → **전이학습**이 필요합니다.

```
5강 Random Forest:                 7강 ResNet-18 전이학습:
  WAV → librosa → 57개 피처        WAV → 멜스펙트로그램 PNG
           ↓                                 ↓
  사람이 설계한 피처 공간            ImageNet 필터(자동 피처 추출)
           ↓                                 ↓
  RandomForest (10클래스)           FC 교체: 512 → 10클래스
```

### 이전 세션과 연결

오늘(7강)에서 배울 것: ImageNet 학습된 ResNet-18의 마지막 FC(1000→10)을 교체하면 장르 분류기가 된다.  
8강에서 배울 것: 실제로 PNG 데이터를 넣어 fine-tuning → 정확도가 RF보다 얼마나 오를까?

> 🔑 **용어: Residual(잔차 연결)이란?**
> 입력을 몇 개 층 건너뛰어 출력에 그대로 더해주는 '지름길'입니다. 깊은 네트워크에서도 신호(gradient)가 사라지지 않게 해줍니다. 아래 다이어그램의 Layer1~4가 각각 이 지름길을 2개씩 포함하고 있습니다(그래서 'ResNet' = Residual Network).

In [23]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 10/16] ResNet-18 아키텍처 요약 다이어그램          │
# │ matplotlib으로 블록 다이어그램 시각화                  │
# │ (실제 torch forward 없이 구조만 보여줌)                │
# └────────────────────────────────────────────────────────┘
# [왜] ResNet-18의 층 구조를 시각적으로 보여줘야 "어디를 교체하는가"가 명확해짐

fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(0, 13)
ax.set_ylim(-0.5, 3)
ax.axis('off')

# [용어] Residual(잔차 연결)이란? — 입력을 몇 개 층 건너뛰어 출력에 그대로 더해주는 지름길입니다.
#        깊은 네트워크에서도 신호(gradient)가 사라지지 않게 해줍니다. 아래 Layer1~4가 각 2개씩 포함합니다.
blocks = [
    (0.3, 'Input\n(3,224,224)',    COLORS['neutral'],   'PNG 멜스펙트로그램'),
    (1.5, 'Conv1\n7×7, 64',        COLORS['primary'],   '저수준 엣지 검출'),
    (3.0, 'Layer1\n2 Residual',    COLORS['primary'],   '텍스처 패턴'),
    (4.5, 'Layer2\n2 Residual',    COLORS['primary'],   '중간 패턴'),
    (6.0, 'Layer3\n2 Residual',    COLORS['secondary'], '고수준 패턴'),
    (7.5, 'Layer4\n2 Residual',    COLORS['secondary'], '장르 관련 패턴'),
    (9.0, 'AvgPool\n(512,1,1)',    COLORS['accent'],    '전역 평균 풀링'),
    (10.5,'FC\n512→1000',         COLORS['warning'],   '★ 교체 대상'),
    (12.0,'FC\n512→10',           COLORS['accent'],    '장르 10클래스'),
]

In [24]:
# [셀 10 계속] 블록 그리기 + 교체 표시 + 출력
for i, (x, label, color, tooltip) in enumerate(blocks):
    rect = mpatches.FancyBboxPatch(
        (x - 0.55, 0.2), 1.1, 1.6,
        boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='white', alpha=0.85, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, 1.0, label, ha='center', va='center',
            fontsize=7.5, color='white', fontweight='bold')
    ax.text(x, -0.1, tooltip, ha='center', va='top',
            fontsize=7, color=COLORS['neutral'])

    if i < len(blocks) - 1:
        next_x = blocks[i + 1][0]
        ax.annotate('',
                    xy=(next_x - 0.57, 1.0), xytext=(x + 0.57, 1.0),
                    arrowprops=dict(arrowstyle='->', color=COLORS['neutral'], lw=1.5))

# 교체 표시 화살표
ax.annotate('교체!\n(전이학습 핵심)',
            xy=(10.5, 1.9), xytext=(10.5, 2.8),
            ha='center', fontsize=9, color=COLORS['warning'], fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLORS['warning'], lw=2))

ax.set_title('ResNet-18 아키텍처 — ImageNet FC(512→1000)를 장르 FC(512→10)으로 교체',
             fontsize=12, fontweight='bold', pad=8)
plt.tight_layout()
plt.show()
print('\n[핵심] 파란·보라 블록은 ImageNet 가중치 유지, 초록 FC만 새로 학습')


[핵심] 파란·보라 블록은 ImageNet 가중치 유지, 초록 FC만 새로 학습


> 📐 **코드를 읽기 전에 — 아래 두 셀이 하는 일**

| 코드 | 하는 일 | 비유 |
|---|---|---|
| `conv1_weights[idx].permute(1, 2, 0)` | (채널,높이,너비) 순서를 (높이,너비,채널)로 재배열 | PyTorch가 계산하기 편한 순서를 우리가 그림으로 보기 편한 순서로 다시 정렬 |
| `.mean(axis=2)` | RGB 3채널을 흑백 1채널로 평균내 압축 | 컬러 사진을 흑백 사진으로 바꿔 무늬(패턴)만 남김 |
| `(f_gray - min) / (max - min)` | 값 범위를 0~1로 맞춤(min-max 정규화) | 사진 밝기를 표준 범위로 맞춰 전부 같은 기준으로 비교 가능하게 함 |

In [25]:
# ┌────────────────────────────────────────────────────────┐
# │ [보강 셀 A] ResNet 첫 레이어 필터 시각화              │
# │ 왜: ImageNet으로 학습된 64개 필터가 어떤 패턴을       │
# │     감지하는지 직접 보면 '사전학습'의 의미가 체감됨  │
# │ 5강 연결: MFCC가 '사람이 설계한 필터'라면,            │
# │          ResNet 필터는 '데이터가 학습한 필터'         │
# └────────────────────────────────────────────────────────┘
# [왜] ResNet의 conv1(첫 번째 합성곱 레이어)는 64개 필터(각 7×7×3)를 가집니다.
#      ImageNet으로 학습했을 때 이 필터들은 에지·색상·질감 등 저수준 패턴을 감지합니다.
#      멜스펙트로그램(PNG)을 입력하면 이 필터들이 시간-주파수 패턴에 반응합니다.

try:
    from torchvision.models import resnet18, ResNet18_Weights
    _tmp_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
except Exception:
    import torchvision.models as _m
    _tmp_model = _m.resnet18(pretrained=False)

# conv1 필터 추출: shape (64, 3, 7, 7)
conv1_weights = _tmp_model.conv1.weight.data.cpu()  # (64, 3, 7, 7)
print(f'conv1 필터 shape: {conv1_weights.shape}')  # (64, 3, 7, 7)
print(f'  → 64개 필터, 각 7×7 픽셀, RGB 3채널')
print(f'  → 이 필터들이 멜스펙트로그램의 시간-주파수 패턴을 감지합니다')

conv1 필터 shape: torch.Size([64, 3, 7, 7])
  → 64개 필터, 각 7×7 픽셀, RGB 3채널
  → 이 필터들이 멜스펙트로그램의 시간-주파수 패턴을 감지합니다


In [26]:
# [보강 셀 A 계속] 필터 32개 시각화 (3채널 → 그레이스케일 평균)
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle(
    'ResNet-18 conv1 사전학습 필터 (64개 중 32개)\n'
    '[ 왼쪽: 에지·선 패턴 감지 ] → 멜스펙트로그램에서 하모닉/타악 패턴 감지로 전용',
    fontsize=10, fontweight='bold', y=1.01
)

for idx, ax in enumerate(axes.flat):
    if idx >= 32:
        ax.axis('off')
        continue
    # (3, 7, 7) → (7, 7, 3) → 그레이스케일
    f = conv1_weights[idx].permute(1, 2, 0).numpy()  # (7,7,3)
    f_gray = f.mean(axis=2)                           # (7,7) 그레이스케일
    # 정규화 (시각화용)
    f_norm = (f_gray - f_gray.min()) / (f_gray.max() - f_gray.min() + 1e-8)
    ax.imshow(f_norm, cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_title(f'F{idx}', fontsize=6, pad=1)
    ax.axis('off')

plt.tight_layout()
plt.show()
print()
print('[핵심 직관]')
print('  5강 RandomForest: 사람이 음악 이론 기반으로 설계한 57개 피처(MFCC 포함)')
print('  ResNet conv1: 100만 장 이미지에서 자동 학습된 64개 필터')
print('  → 전이학습 = 이 64개 필터를 멜스펙트로그램에 그대로 재사용')
print()
print('[학생 실험] filter 번호를 바꿔 다른 패턴을 관찰해보세요')
print('  단색(평탄) 필터 vs 줄무늬 필터 — 어떤 음악 특성에 반응할까요?')

del _tmp_model  # 메모리 정리


[핵심 직관]
  5강 RandomForest: 사람이 음악 이론 기반으로 설계한 57개 피처(MFCC 포함)
  ResNet conv1: 100만 장 이미지에서 자동 학습된 64개 필터
  → 전이학습 = 이 64개 필터를 멜스펙트로그램에 그대로 재사용

[학생 실험] filter 번호를 바꿔 다른 패턴을 관찰해보세요
  단색(평탄) 필터 vs 줄무늬 필터 — 어떤 음악 특성에 반응할까요?


> 🔑 **용어: 과적합(overfitting) · Dropout이란?**
> **과적합**은 모델이 연습문제(학습 데이터)를 통째로 외워버려서, 처음 보는 문제(새 데이터)에서는 오히려 성적이 떨어지는 상태입니다. **Dropout**은 학습 중 일부 뉴런을 무작위로 잠깐씩 꺼버리는 장치로, 특정 뉴런에만 의존해서 통암기하지 못하게 방해합니다 — 매번 다른 팀원 조합으로 문제를 풀게 해서 한 사람에게만 의존하지 못하게 하는 것과 비슷합니다.

In [27]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 11/16] load_resnet_model() 함수 정의              │
# │ torchvision.models.resnet18 + 마지막 레이어 교체       │
# │ 입력: 없음   출력: model (10클래스 분류기)             │
# └────────────────────────────────────────────────────────┘
# [왜] pretrained=True → ImageNet 가중치를 재활용
#      model.fc = Linear(512, 10) → 10 장르에 맞게 교체

import torch.nn as nn

def load_resnet_model(n_classes: int = 10, freeze_backbone: bool = False) -> nn.Module:
    """
    ImageNet 사전학습 ResNet-18을 로드하고 마지막 FC를 교체.

    Args:
        n_classes      : 출력 클래스 수 (기본 10 장르)
        freeze_backbone: True → fc만 학습, backbone 동결 (feature extractor 모드)
    Returns:
        nn.Module — 디바이스로 이동된 모델
    """
    try:
        # torchvision >= 0.13 권장 방식
        from torchvision.models import ResNet18_Weights
        model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    except ImportError:
        model = models.resnet18(pretrained=True)  # 구버전 호환

    # backbone 동결 옵션 (★★★ 트랙: 학습 데이터 적을 때)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # 마지막 FC 레이어 교체: 512 → n_classes
    in_features   = model.fc.in_features          # ResNet-18은 512
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),                        # 과적합 방지
        nn.Linear(in_features, n_classes)         # 새 FC
    )

    model = model.to(DEVICE)
    return model

In [28]:
# [셀 11 계속] 모델 생성 + 파라미터 수 확인
resnet_model = load_resnet_model(n_classes=10, freeze_backbone=False)

total_params     = sum(p.numel() for p in resnet_model.parameters())
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f'ResNet-18 (ImageNet → 장르 10클래스):')
print(f'  전체 파라미터:     {total_params:>10,}  ({total_params/1e6:.1f}M)')
print(f'  학습 가능 파라미터: {trainable_params:>10,}')
print(f'  동결된 파라미터:    {frozen_params:>10,}')
print(f'  교체된 FC:          {resnet_model.fc}')
print(f'  디바이스:           {next(resnet_model.parameters()).device}')

ResNet-18 (ImageNet → 장르 10클래스):
  전체 파라미터:     11,181,642  (11.2M)
  학습 가능 파라미터: 11,181,642
  동결된 파라미터:             0
  교체된 FC:          Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=512, out_features=10, bias=True)
)
  디바이스:           mps:0


> **🔍 ResNet-18 파라미터 결과 해석**
> - 전체 11M 파라미터 중 FC(Dropout + Linear)는 ~5K — 나머지는 ImageNet 학습 그대로
> - freeze_backbone=True이면 FC 5K만 학습 → 데이터 적어도 과적합 낮음 (feature extractor 모드)
> - 8강에서는 freeze_backbone=False로 전체를 fine-tuning하여 성능 차이를 비교함
> - **왜 데이터가 적을 때 유리한가**: 방금(위 과적합·Dropout 개념) 배웠듯, 파라미터 수(11M)가 학습 데이터 수(GTZAN 940장)보다 압도적으로 많으면 모델이 데이터를 이해하는 대신 통째로 외워버릴 위험이 큽니다. `freeze_backbone=True`는 학습 대상을 FC의 ~5K 파라미터로 줄여 데이터 대비 파라미터 비율을 훨씬 안전한 범위로 낮춥니다.
> - 🔭 이 비교는 AI Pair 섹션 **Solo 레벨 2(★★ 심화 트랙)**에서 `freeze_backbone=True`로 직접 모델을 만들고 위와 똑같은 `trainable_params` 계산을 다시 실행해 지금 이 결과(`False`)와 나란히 비교하는 과제로 이어집니다.

In [29]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 12/16] RUN_HEAVY 게이트 — PNG 추론 실행 여부 결정 │
# │ ★ CPU 트랙: 이 셀에서 멈추고 결과만 확인              │
# │ ★★ GPU 트랙: 아래 셀에서 실제 추론 실행              │
# └────────────────────────────────────────────────────────┘
print(f'디바이스: {DEVICE}  |  RUN_HEAVY: {RUN_HEAVY}')

if not RUN_HEAVY:
    print()
    print('=' * 58)
    print('  CPU 환경 → ★★ ResNet-18 추론 스킵 (시뮬레이션 결과)')
    print('=' * 58)
    print()
    print('[시뮬레이션] 단일 PNG 추론 결과:')
    print('  입력 이미지: classical.00000.png')
    print('  예측 장르  : classical (확률 0.921)')
    print('  추론 시간  : 0.03초 (GPU T4 기준)')
    print()
    print('GPU 환경에서 실행하려면:')
    print('  RUN_HEAVY = True  # 이 줄을 직접 설정하거나')
    print('  Google Colab → 런타임 > 런타임 유형 변경 > GPU')
else:
    print('\nGPU/MPS 확인 완료 — 아래 셀에서 PNG 추론을 실행합니다')

디바이스: mps  |  RUN_HEAVY: True

GPU/MPS 확인 완료 — 아래 셀에서 PNG 추론을 실행합니다


> 🔑 **용어: logits란?**
> 모델이 마지막에 내놓는 '정규화 전 원점수'(클래스별 선호도 점수)입니다. `torch.softmax(logits, dim=-1)`가 이 점수를 우리가 아는 0~1 사이 확률로 바꿔줍니다 — 점수가 큰 클래스일수록 확률도 높아집니다.

In [30]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 13/16] ★★ PNG 스펙트로그램 단일 추론              │
# │ transform 적용 → ResNet 추론 → softmax 확률 출력      │
# └────────────────────────────────────────────────────────┘
# [왜] 학생이 transform 파이프라인의 각 단계를 shape로 추적하게 함

from PIL import Image

# ImageNet 정규화 (전이학습 시 반드시 적용)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),          # ResNet 표준 입력 크기
    transforms.ToTensor(),                  # [0,255] → [0.0,1.0] + CHW
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def infer_single_png(img_path, model: nn.Module, label_names: list) -> tuple:
    """
    단일 PNG 이미지 추론.
    Returns: (예측장르, 확률dict)
    """
    img   = Image.open(str(img_path)).convert('RGB')
    x     = infer_transform(img).unsqueeze(0).to(DEVICE)  # 원본의 model.device는 nn.Module에 없는 속성 — DEVICE로 수정
    model.eval()
    with torch.no_grad():
        # [용어] logits = 정규화 전 원점수(클래스별 '선호도' 점수) — softmax가 이를 0~1 확률로 변환
        logits = model(x)                          # (1, 10)
        proba  = torch.softmax(logits, dim=-1)[0]  # (10,)
    proba_np = proba.cpu().numpy()
    prob_dict = {label_names[i]: float(proba_np[i]) for i in range(len(label_names))}
    prob_dict = dict(sorted(prob_dict.items(), key=lambda x: x[1], reverse=True))
    return list(prob_dict.keys())[0], prob_dict

In [31]:
# [셀 13 계속] 추론 실행 (RUN_HEAVY 게이트)
if RUN_HEAVY and PNG_ROOT.exists():
    _sample_png = PNG_ROOT / 'classical' / 'classical00000.png'
    if not _sample_png.exists():
        # 디렉토리 내 첫 번째 PNG
        _sample_png = next((PNG_ROOT / 'classical').glob('*.png'), None)

    if _sample_png and _sample_png.exists():
        t0 = time.time()
        _pred_g, _pred_p = infer_single_png(_sample_png, resnet_model, GENRES)
        _elapsed = time.time() - t0
        print(f'입력 이미지: {_sample_png.name}')
        print(f'추론 시간  : {_elapsed:.3f}초')
        print(f'예측 장르  : {_pred_g}  (확률 {_pred_p[_pred_g]:.3f})')
        print('\n전체 확률:')
        for g, p in list(_pred_p.items())[:5]:
            print(f'  {g:<12} {p:.4f}')
    else:
        print('[주의] classical PNG 파일 없음 — PNG_ROOT를 확인하세요')
        print(f'  PNG_ROOT: {PNG_ROOT}')
elif RUN_HEAVY and not PNG_ROOT.exists():
    print(f'[주의] PNG_ROOT 없음: {PNG_ROOT}')
    print('  images_original 폴더가 있어야 합니다')
else:
    print('[SKIP] CPU 환경 → 셀 12의 시뮬레이션 결과를 참고하세요')

입력 이미지: classical00000.png
추론 시간  : 0.645초
예측 장르  : country  (확률 0.169)

전체 확률:
  country      0.1686
  pop          0.1500
  metal        0.1387
  jazz         0.1249
  reggae       0.1030


> **🔍 결과 해석 — classical PNG가 왜 country(16.9%)로 나왔을까?**
> - FC(Fully Connected, 완전연결층 — 마지막 레이어)를 **방금 무작위로 교체**했고, 아직 8강의 fine-tuning을 거치지 않았습니다 — conv1~layer4는 ImageNet 가중치 그대로지만, 새 FC(512→10)는 학습된 적이 없는 초기화 상태입니다.
> - 그래서 10개 장르 확률이 country 0.169, pop 0.150, metal 0.139, jazz 0.125, reggae 0.103처럼 1/10=0.10 근처에 넓게 흩어져 나오는 게 **정상**입니다 — 모델이 아직 아무것도 배우지 않았기 때문입니다.
> - 이걸 고치는 것이 바로 8강의 fine-tuning입니다: 이 새 FC(그리고 필요하면 상위 레이어)를 실제 멜스펙트로그램 PNG로 학습시켜야 country 0.169 같은 무작위에 가까운 확률이 classical 0.9대로 올라갑니다.
> - **참고(속도)**: 위 [셀 12/16]의 시뮬레이션 값(0.03초)과 실제 추론 시간이 크게 다르게 나올 수 있습니다 — GPU/MPS는 첫 연산 시 내부 초기화(웜업)에 몇 초가 걸리고, 두 번째 추론부터는 훨씬 빨라집니다. 느리게 나왔다고 환경이 잘못된 것이 아니라, **처음 한 번만 그런 것**이니 걱정하지 마세요.
> - 🔭 이 단일 PNG 추론 과정은 AI Pair 섹션 **Solo 레벨 2(★★ 심화 트랙)** 1번 문제에서 `freeze_backbone=True`/`False` 두 모드로 다시 실행해 비교하는 데 그대로 재사용됩니다.

---
## 섹션 4 마무리 — RF vs ResNet-18 비교표

| 항목 | RandomForest (★) | ResNet-18 (★★) |
|------|-----------------|----------------|
| 입력 | librosa 57개 피처 | 멜스펙트로그램 PNG (224×224) |
| 피처 설계 | 사람이 직접 | 모델이 자동 학습 |
| 학습 시간 | 수초 (CPU) | 수십 분 (GPU 권장) |
| 새 곡 적용 | librosa 재추출 필요 | PNG 변환 후 바로 추론 |
| 데이터 요구 | 적어도 가능 | 전이학습으로 극복 |
| 정확도 (GTZAN) | ~90% | ~88~93% (fine-tuning 후) |
| 배포 크기 | ~수 MB | ~수십 MB |
| 적합 상황 | 빠른 프로토타입, CPU 환경 | 이미지 스펙트로그램, 새 도메인 |

**→ 실무 결론:** RF는 빠른 검증용, ResNet-18은 서비스 품질 확보용. 8강에서 실제 fine-tuning 후 수치로 재검토.

In [32]:
# ┌────────────────────────────────────────────────────────┐
# │ [보강 셀 B] RF 피처 중요도 vs ResNet 예측 신뢰도 비교 │
# │ 왜: 비교표로 끝나지 않고 실제 수치로 체험해야         │
# │     '그래서 어느 쪽이 더 나은가'가 와닿음             │
# └────────────────────────────────────────────────────────┘
# [왜] RF는 '어떤 피처가 중요했는가'를 feature_importances_로 설명할 수 있습니다.
#      ResNet은 픽셀 전체를 보지만 어떤 주파수 영역이 결정적이었는지 직접 보기 어렵습니다.
#      이 '설명가능성(interpretability) 트레이드오프'가 RF vs DeepNet의 핵심 차이입니다.

# ── RF 피처 중요도 상위 15개 시각화 ─────────────────────
importances = rf.feature_importances_          # (57,)
feat_imp_df = pd.DataFrame({
    '피처': FEATURE_COLS,
    '중요도': importances
}).sort_values('중요도', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: RF 피처 중요도
bar_colors_imp = [COLORS['primary']] * 5 + [COLORS['secondary']] * 5 + [COLORS['neutral']] * 5
axes[0].barh(
    feat_imp_df['피처'].values[::-1],
    feat_imp_df['중요도'].values[::-1],
    color=bar_colors_imp[::-1]
)
axes[0].set_title(
    'RF 피처 중요도 Top 15\n(설명가능: 어떤 피처가 결정적이었나)',
    fontsize=10, fontweight='bold'
)
axes[0].set_xlabel('Importance')
axes[0].tick_params(axis='y', labelsize=8)

In [33]:
# [보강 셀 B 계속] RF vs ResNet-18 속성 비교 막대 + 결과 출력

# 오른쪽: RF vs ResNet-18 속성 비교 막대 (스코어 기반)
categories = ['설명가능성', '데이터\n효율', '피처\n자동화', '새 도메인\n적응', 'CPU\n속도']
rf_scores   = [9, 8, 2, 3, 10]    # RF 강점
resnet_scores = [4, 5, 9, 9, 3]   # ResNet 강점

x = range(len(categories))
width = 0.35
axes[1].bar([xi - width/2 for xi in x], rf_scores,
            width=width, label='RandomForest', color=COLORS['primary'], alpha=0.85)
axes[1].bar([xi + width/2 for xi in x], resnet_scores,
            width=width, label='ResNet-18', color=COLORS['secondary'], alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(categories, fontsize=9)
axes[1].set_ylim(0, 12)
axes[1].set_ylabel('상대 점수 (10점 만점)')
axes[1].set_title(
    'RF vs ResNet-18 속성 레이더\n→ 실무: RF(프로토타입) → ResNet(서비스 품질)',
    fontsize=10, fontweight='bold'
)
axes[1].legend(fontsize=9)
axes[1].axhline(y=5, color='gray', linestyle='--', alpha=0.4, label='기준선')

plt.tight_layout()
plt.show()

print()
print('─' * 55)
print('RF Top 5 중요 피처 (5강에서 배운 피처들!):')
for _, row in feat_imp_df.head(5).iterrows():
    bar = '█' * int(row['중요도'] * 200)
    print(f'  {row["피처"]:<30} {row["중요도"]:.4f}  {bar}')
print()
print('→ 실무 결론:')
print('  [프로토타입] RF: 피처 중요도로 왜 그 장르인지 설명 가능 (explainable)')
print('  [서비스 품질] ResNet: 픽셀 자체를 학습 — 8강 fine-tuning 후 정확도 비교 예정')
print()
print('[8강 예고] fine-tuning 전/후 ResNet 정확도가 RF를 넘는 순간을 직접 확인합니다!')


───────────────────────────────────────────────────────
RF Top 5 중요 피처 (5강에서 배운 피처들!):
  perceptr_var                   0.0524  ██████████
  chroma_stft_mean               0.0404  ████████
  rms_mean                       0.0361  ███████
  rms_var                        0.0340  ██████
  mfcc4_mean                     0.0303  ██████

→ 실무 결론:
  [프로토타입] RF: 피처 중요도로 왜 그 장르인지 설명 가능 (explainable)
  [서비스 품질] ResNet: 픽셀 자체를 학습 — 8강 fine-tuning 후 정확도 비교 예정

[8강 예고] fine-tuning 전/후 ResNet 정확도가 RF를 넘는 순간을 직접 확인합니다!


---
<a id='sec5'></a>
## 섹션 5 — app_v2.py 생성: 신뢰도 + 이력 + 멀티파일 통합

### 무엇이 달라지는가? (app.py → app_v2.py)

```
[app.py (6강)]                    [app_v2.py (7강)]
  단일 파일 업로드                  accept_multiple_files=True
  1위 장르만 표시                   신뢰도 bar chart (10장르 전체)
  이력 없음                         st.session_state['history'] 테이블
  비교 불가                          grouped bar chart 비교
```

In [34]:
%%writefile "{NB_DIR}/app_v2.py"
# ┌────────────────────────────────────────────────────────┐
# │ [셀 14/16] app_v2.py 저장 (섹션 1/9)                    │
# └────────────────────────────────────────────────────────┘
# 7강 실습 — Streamlit 장르 예측 앱 v2
# 신뢰도 bar chart + session_state 이력 + 멀티파일 비교
# 사용법: streamlit run app_v2.py

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
import librosa.display
import joblib
import tempfile, os, platform

Overwriting app_v2.py


In [35]:
%%writefile -a "{NB_DIR}/app_v2.py"
# ── 한글 폰트 설정 ─────────────────────────────────────
from pathlib import Path
import matplotlib.font_manager as fm

def setup_korean_font():
    system = platform.system()
    preferred = {
        "Darwin": ["AppleGothic", "Apple SD Gothic Neo", "NanumGothic", "Noto Sans CJK KR"],
        "Windows": ["Malgun Gothic", "맑은 고딕", "NanumGothic", "Noto Sans CJK KR"],
        "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans KR", "Noto Sans CJK JP"],
    }
    font_files = {
        "Darwin": [
            "/System/Library/Fonts/AppleGothic.ttf",
            "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
            str(Path.home() / "Library/Fonts/NanumGothic.ttf"),
        ],
        "Windows": ["C:/Windows/Fonts/malgun.ttf"],
        "Linux": [
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
            "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
            "/usr/share/fonts/truetype/noto/NotoSansKR-Regular.otf",
        ],
    }
    for font_path in font_files.get(system, []):
        if Path(font_path).exists():
            fm.fontManager.addfont(font_path)
    available = {f.name for f in fm.fontManager.ttflist}
    chosen = next((font for font in preferred.get(system, []) if font in available), None)
    # Streamlit Cloud 배포(Linux) 환경엔 한글 폰트가 기본 설치돼 있지 않을 수 있어,
    # 없으면 apt-get으로 나눔고딕을 설치한 뒤 다시 탐색합니다.
    if chosen is None and system == "Linux":
        import os
        os.system("apt-get -qq -y install fonts-nanum > /dev/null 2>&1")
        for _fp in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
            fm.fontManager.addfont(_fp)
        available = {f.name for f in fm.fontManager.ttflist}
        chosen = next((font for font in preferred.get(system, []) if font in available), None)
    if chosen is None:
        chosen = "DejaVu Sans"
    matplotlib.rcParams["font.family"] = chosen
    matplotlib.rcParams["font.sans-serif"] = [chosen, "NanumGothic", "Noto Sans CJK KR", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    return chosen

setup_korean_font()

Appending to app_v2.py


In [36]:
%%writefile -a "{NB_DIR}/app_v2.py"
FEATURE_COLS = (
    ["chroma_stft_mean","chroma_stft_var",
     "rms_mean","rms_var",
     "spectral_centroid_mean","spectral_centroid_var",
     "spectral_bandwidth_mean","spectral_bandwidth_var",
     "rolloff_mean","rolloff_var",
     "zero_crossing_rate_mean","zero_crossing_rate_var",
     "harmony_mean","harmony_var",
     "perceptr_mean","perceptr_var",
     "tempo"]
    + [f"mfcc{i}_{s}" for i in range(1, 21) for s in ("mean", "var")]
)
GENRES = ["blues","classical","country","disco",
          "hiphop","jazz","metal","pop","reggae","rock"]
COLORS = {
    "primary":   "#2563EB",
    "secondary": "#7C3AED",
    "accent":    "#059669",
    "neutral":   "#6B7280",
    "warning":   "#F59E0B",
}

Appending to app_v2.py


In [37]:
%%writefile -a "{NB_DIR}/app_v2.py"
@st.cache_resource
def load_models():
    base = os.path.dirname(os.path.abspath(__file__))
    rf = joblib.load(os.path.join(base, "model_rf.joblib"))
    le = joblib.load(os.path.join(base, "label_encoder.joblib"))
    sc = joblib.load(os.path.join(base, "scaler.joblib"))
    return rf, le, sc

def extract_features(wav_path):
    y, sr = librosa.load(wav_path, sr=22050, mono=True, duration=3.0)
    feats = {}
    ch = librosa.feature.chroma_stft(y=y, sr=sr)
    feats["chroma_stft_mean"] = float(np.mean(ch)); feats["chroma_stft_var"] = float(np.var(ch))
    rm = librosa.feature.rms(y=y)
    feats["rms_mean"] = float(np.mean(rm)); feats["rms_var"] = float(np.var(rm))
    sc2 = librosa.feature.spectral_centroid(y=y, sr=sr)
    feats["spectral_centroid_mean"] = float(np.mean(sc2)); feats["spectral_centroid_var"] = float(np.var(sc2))
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    feats["spectral_bandwidth_mean"] = float(np.mean(bw)); feats["spectral_bandwidth_var"] = float(np.var(bw))
    ro = librosa.feature.spectral_rolloff(y=y, sr=sr)
    feats["rolloff_mean"] = float(np.mean(ro)); feats["rolloff_var"] = float(np.var(ro))
    zcr = librosa.feature.zero_crossing_rate(y)
    feats["zero_crossing_rate_mean"] = float(np.mean(zcr)); feats["zero_crossing_rate_var"] = float(np.var(zcr))
    harm, perc = librosa.effects.hpss(y)
    feats["harmony_mean"] = float(np.mean(harm)); feats["harmony_var"] = float(np.var(harm))
    feats["perceptr_mean"] = float(np.mean(perc)); feats["perceptr_var"] = float(np.var(perc))
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    feats["tempo"] = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])
    mf = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    for i in range(20):
        feats[f"mfcc{i+1}_mean"] = float(np.mean(mf[i]))
        feats[f"mfcc{i+1}_var"]  = float(np.var(mf[i]))
    return np.array([feats[c] for c in FEATURE_COLS], dtype=np.float32).reshape(1, -1)

Appending to app_v2.py


In [38]:
%%writefile -a "{NB_DIR}/app_v2.py"
def predict_with_confidence(wav_path, rf, le, sc):
    vec    = extract_features(wav_path)
    vec_sc = sc.transform(vec)
    proba  = rf.predict_proba(vec_sc)[0]
    prob_dict = {le.classes_[i]: float(proba[i]) for i in range(len(proba))}
    prob_dict = dict(sorted(prob_dict.items(), key=lambda x: x[1], reverse=True))
    return list(prob_dict.keys())[0], prob_dict

def plot_confidence_bar(prob_dict, title=""):
    genres = list(prob_dict.keys())
    probs  = list(prob_dict.values())
    top_g  = genres[0]
    colors = [COLORS["primary"] if g == top_g else COLORS["neutral"] for g in genres]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(genres[::-1], probs[::-1], color=colors[::-1],
            edgecolor="white", height=0.55, alpha=0.88)
    for g, p in zip(genres[::-1], probs[::-1]):
        if p > 0.01:
            ax.text(p + 0.01, genres[::-1].index(g), f"{p:.1%}", va="center", fontsize=8)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("확률")
    ax.set_title(title or f"1위: {top_g.upper()}", fontsize=11)
    ax.axvline(0.5, color=COLORS["warning"], lw=1.2, linestyle="--", label="50% 기준")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

Appending to app_v2.py


In [39]:
%%writefile -a "{NB_DIR}/app_v2.py"
def plot_melspectrogram(wav_path, title=""):
    y, sr = librosa.load(wav_path, sr=22050, mono=True, duration=10.0)
    mel   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    fig, ax = plt.subplots(figsize=(6, 3))
    img = librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel",
                                    fmax=8000, ax=ax, cmap="magma")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    ax.set_title(title or "멜스펙트로그램", fontsize=10)
    fig.tight_layout()
    return fig

# ══════════════════════════════════════════════════════════
# Streamlit UI
# ══════════════════════════════════════════════════════════
st.set_page_config(page_title="장르 예측 v2", page_icon="🎵", layout="wide")
st.title("🎵 장르 예측기 v2 — 신뢰도 + 이력 + 멀티파일 비교")
st.caption("AI Human 개발자 과정 강사 김생근 · 7강 실습 — 6강 앱 업그레이드")

# 이력 초기화
if "history" not in st.session_state:
    st.session_state["history"] = []

with st.spinner("모델 로딩..."):
    try:
        rf_m, le_m, sc_m = load_models()
        st.success("모델 로드 완료", icon="✅")
    except FileNotFoundError as e:
        st.error(f"모델 파일 없음: {e}")
        st.stop()

Appending to app_v2.py


In [40]:
%%writefile -a "{NB_DIR}/app_v2.py"
tab1, tab2 = st.tabs(["🎵 예측", "📋 이력"])

with tab1:
    uploaded_files = st.file_uploader(
        "WAV 파일 업로드 (여러 개 가능)",
        type=["wav"],
        accept_multiple_files=True,
        help="7강: 여러 곡을 동시에 업로드해 장르 분포를 비교하세요"
    )

    if uploaded_files:
        tmp_paths = []
        for uf in uploaded_files:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp.write(uf.read())
                tmp_paths.append((uf.name, tmp.name))

        if len(tmp_paths) == 1:
            fname, fpath = tmp_paths[0]
            st.audio(fpath, format="audio/wav")
            top_g, probs = predict_with_confidence(fpath, rf_m, le_m, sc_m)
            col1, col2 = st.columns([1, 1])
            with col1:
                st.subheader("신뢰도 분포")
                fig_bar = plot_confidence_bar(probs, title=fname)
                st.pyplot(fig_bar); plt.close(fig_bar)
            with col2:
                st.subheader("멜스펙트로그램")
                fig_mel = plot_melspectrogram(fpath, title=fname)
                st.pyplot(fig_mel); plt.close(fig_mel)
            # 이력 추가
            st.session_state["history"].append({
                "파일명": fname, "1위 장르": top_g,
                "1위 확률": f"{probs[top_g]:.1%}",
                "2위 장르": list(probs.keys())[1],
                "고신뢰도": "✓" if probs[top_g] >= 0.5 else "△",
            })

Appending to app_v2.py


In [41]:
%%writefile -a "{NB_DIR}/app_v2.py"
        else:
            # 멀티파일 비교
            st.subheader(f"{len(tmp_paths)}개 곡 비교")
            all_probs_list = []
            result_rows = []
            for fname, fpath in tmp_paths:
                top_g, probs = predict_with_confidence(fpath, rf_m, le_m, sc_m)
                all_probs_list.append([probs.get(g, 0) for g in GENRES])
                result_rows.append({"파일명": fname, "1위 장르": top_g,
                                     "1위 확률": f"{probs[top_g]:.1%}"})
                st.session_state["history"].append({
                    "파일명": fname, "1위 장르": top_g,
                    "1위 확률": f"{probs[top_g]:.1%}",
                    "2위 장르": list(probs.keys())[1],
                    "고신뢰도": "✓" if probs[top_g] >= 0.5 else "△",
                })
            st.dataframe(pd.DataFrame(result_rows))
            # grouped bar chart
            import numpy as np_inner
            x = np_inner.arange(len(GENRES))
            width = 0.8 / len(tmp_paths)
            palette = ["#2563EB","#7C3AED","#059669","#F59E0B"]
            fig_cmp, ax_cmp = plt.subplots(figsize=(12, 4))
            for i, (probs_row, (fname, _)) in enumerate(zip(all_probs_list, tmp_paths)):
                offset = (i - len(tmp_paths)/2 + 0.5) * width
                ax_cmp.bar(x + offset, probs_row, width,
                           label=fname, color=palette[i % len(palette)], alpha=0.82)
            ax_cmp.set_xticks(x); ax_cmp.set_xticklabels(GENRES, rotation=30)
            ax_cmp.set_ylabel("확률"); ax_cmp.set_ylim(0, 1.05)
            ax_cmp.set_title("멀티파일 장르 확률 비교"); ax_cmp.legend()
            ax_cmp.grid(axis="y", alpha=0.3)
            fig_cmp.tight_layout()
            st.pyplot(fig_cmp); plt.close(fig_cmp)

        for _, fpath in tmp_paths:
            os.unlink(fpath)

Appending to app_v2.py


In [42]:
%%writefile -a "{NB_DIR}/app_v2.py"
with tab2:
    st.subheader("예측 이력")
    if st.session_state["history"]:
        df_h = pd.DataFrame(st.session_state["history"])
        st.dataframe(df_h, use_container_width=True)
        if st.button("이력 초기화"):
            st.session_state["history"] = []
            st.rerun()
    else:
        st.info("아직 예측 이력이 없습니다. [예측] 탭에서 파일을 업로드하세요.")

Appending to app_v2.py


In [43]:
# app_v2.py 문법 검증 (SyntaxError 재발 방지) — NB_DIR(AI_Music 루트)에 저장된 파일을 확인합니다.
import ast
with open(NB_DIR / 'app_v2.py', encoding='utf-8') as f:
    _src = f.read()
ast.parse(_src)
print('app_v2.py SYNTAX OK')

app_v2.py SYNTAX OK


In [44]:
# ┌────────────────────────────────────────────────────────┐
# │ [셀 15/16] 로컬 실행 안내 + 파일 구조 확인            │
# └────────────────────────────────────────────────────────┘
print('현재 폴더 파일 목록:')
for p in sorted(NB_DIR.iterdir()):
    if not p.name.startswith('.'):
        size = p.stat().st_size
        print(f'  {p.name:<40} {size:>8,} B')

print()
print('=' * 58)
print('  app_v2.py 실행 방법')
print('=' * 58)
print(f'  cd "{NB_DIR}"')
print('  streamlit run app_v2.py')
print()
print('  브라우저: http://localhost:8501')
print()
print('  필요 파일 체크:')
for f in ['model_rf.joblib', 'label_encoder.joblib', 'scaler.joblib', 'app_v2.py']:
    exists = (NB_DIR / f).exists()
    print(f'    ["{"OK" if exists else "없음"}"] {f}')

현재 폴더 파일 목록:
  1강_내노래를_숫자로보기(AI_Pair).ipynb             104,932,327 B
  2강_요즘음악_에너지템포비교(AI_Pair).ipynb            259,393 B
  3강_플레이리스트_클러스터링(AI_Pair).ipynb            135,789 B
  4강_소리를이미지로_멜스펙트로그램(AI_Pair).ipynb        3,806,884 B
  5강_멜스펙트로그램_장르분류(AI_Pair).ipynb           2,285,397 B
  6강_Streamlit_장르예측앱(AI_Pair).ipynb         121,882 B
  7강_Streamlit고급_ResNet도입(AI_Pair).ipynb    141,014 B
  8강_ResNet18_스펙트로그램분류(AI_Pair).ipynb      1,221,585 B
  Data                                          160 B
  README.md                                   3,237 B
  _edit_lecture3_ux_fixes.py                  3,761 B
  _edit_lecture6.py                          11,943 B
  _edit_lecture6_finalize.py                  4,073 B
  _edit_lecture8_round2.py                   36,491 B
  app.py                                      9,953 B
  app_preview.py                              2,338 B
  app_v2.py                                  11,654 B
  d8_fallback_data                              128 B
  label_e

---
<a id='fin'></a>
## 🤝 AI Pair 섹션 — 신뢰도·이력·전이학습 개념 정리

> **목표**: AI를 답 베끼는 도구가 아니라 *내 코드·판단을 검증해주는 동료*로 쓴다.

| 단계 | 내가 하는 것 | AI가 하는 것 |
|---|---|---|
| 1️⃣ Solo | 먼저 직접 작성/판단 | (아직 X) |
| 2️⃣ Review | 내 코드·판단을 제출 | 리뷰 + 근거 설명 |
| 3️⃣ Debug | AI가 준 "조용히 틀린" 코드의 결함 찾기 | 의도적 버그 제공 |
| 4️⃣ Prompt Card | 실험 설계 프롬프트 익히기 | — |

### 1️⃣ Solo — 먼저 스스로 풀어보세요

**★ 기본 트랙 (모든 학생)**
1. `app_v2.py`를 실행하고 **3곡 이상** 업로드하세요.
2. 신뢰도 bar chart 스크린샷을 캡처하세요.
3. 확률 분포가 고른 곡과 몰린 곡 각 1개를 찾아 이유를 1줄로 서술하세요.

**★★ 심화 트랙 (GPU 보유)**
1. `load_resnet_model(freeze_backbone=True)`와 `False` 두 가지 모드로 단일 PNG 추론 결과를 비교하세요.
2. 두 모드의 파라미터 수(trainable) 차이를 표로 정리하세요.
3. 왜 `freeze_backbone=True` 모드가 학습 데이터가 적을 때 유리한지 설명하세요.

> **🧭 트랙 ↔ 아래 코드 셀 매핑** — 어느 레벨이 내 과제인지 헷갈린다면 이 표를 보세요.
>
> | 아래 코드 셀 | 해당 트랙 | GPU 필요? |
> |---|---|---|
> | [Solo 레벨 1] 신뢰도 bar chart (rock·hiphop) | ★ 기본 트랙 항목 1~3의 확장 | 불필요 — 전원 필수 |
> | [Solo 레벨 2] freeze_backbone 비교 | ★★ 심화 트랙 항목 1~3 그대로 | **GPU 보유 학생 전용.** CPU 환경이면 파라미터 수 비교(구조 이해)까지만 실행하고 추론 속도 비교는 생략해도 됩니다 |
> | [Solo 레벨 3] 확신도 점수 함수 + 요약표 | ★ 기본 트랙의 심화 확장 | 불필요 — `predict_with_confidence`(RandomForest)만 사용해 전원 수행 가능 |

### ✏️ [Solo 레벨 1] 직접 작성해 보세요 — rock, hiphop 곡으로 신뢰도 bar chart 그리기

힌트: predict_with_confidence(wav_path) → plot_confidence_bar(prob_dict) 순서로 호출하면 됩니다.
jazz(위 예시)와 비교해 어느 장르가 더 확신도가 높은지 표준편차로도 확인하세요.

In [45]:
# TODO: 여기에 작성하세요

### ✏️ [Solo 레벨 2] 직접 작성해 보세요 — freeze_backbone=True로 모델을 만들고 파라미터 수 비교

힌트: load_resnet_model(n_classes=10, freeze_backbone=True) 호출 후
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) 로 계산
freeze_backbone=False(원본 셀)의 값과 나란히 출력하세요.

In [46]:
# TODO: 여기에 작성하세요

### ✏️ [Solo 레벨 3] 직접 작성해 보세요 — 확신도 점수 함수 + 전체 곡 요약표

힌트:
1) def confidence_score(prob_dict): return np.std(list(prob_dict.values())) 같은 함수를 만드세요.
2) test_files 전체 곡 각각에 적용해 {파일명: 확신도} 딕셔너리를 만드세요.
3) 확신도가 가장 낮은(=헷갈리는) 곡의 1위 장르와 2위 장르(prob_dict에서 정렬 후 두 번째 값)를 비교해
어떤 장르 쌍이 자주 헷갈리는지 서술하세요. (참고: 이 test_files는 전부 1위 예측이 실제 장르와 일치하므로
'실제 vs 예측'이 아니라 '1위 확률 vs 2위 확률'로 헷갈림을 판단합니다 — 위 섹션 2 결과 해석의 힌트를 참고하세요)

In [47]:
def confidence_score(prob_dict):
    # TODO: 여기에 작성하세요
    pass

### 2️⃣ Review
아래 프롬프트를 복사해 ChatGPT/Claude에 붙여넣으세요.
```text
Streamlit 장르 예측 앱에 신뢰도 bar chart와 st.session_state 이력을 추가했습니다.
predict_with_confidence()는 RandomForest.predict_proba()의 10개 확률을 정렬해 반환합니다.
1) 왜 1위 장르만 보여주는 것보다 확률 분포 전체를 보여주는 게 사용자에게 더 정직한지,
2) st.session_state가 왜 필요한지(Streamlit의 재실행 모델과 연결해서),
3) freeze_backbone=True와 False 중 GTZAN처럼 데이터가 940장뿐인 상황엔 어느 쪽이 더 안전한지
각각 근거를 들어 설명해줘.
```

In [48]:
# ── 2️⃣ Review (코드형) — 로컬/클라우드 LLM에게 내 코드·판단을 리뷰받기 ──
# [사전조건] 로컬: LM Studio/Ollama 서버 실행  |  클라우드: OPENROUTER/OPENAI 키 설정
import os
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

if OpenAI is None:
    print("[안내] openai 패키지가 없어 이 Review 셀을 건너뜁니다 — `!pip install openai` 후 다시 실행하세요(선택 사항입니다).")
else:
    PROVIDER = "lmstudio"   # "lmstudio" | "ollama" | "openrouter" | "openai"  ← 한 줄만 바꾸면 전환
    PROVIDERS = {
        "lmstudio":   {"base_url": "http://localhost:1234/v1",  "model": "local-model",          "api_key": "lm-studio"},
        "ollama":     {"base_url": "http://localhost:11434/v1", "model": "llama3.2",              "api_key": "ollama"},
        "openrouter": {"base_url": "https://openrouter.ai/api/v1", "model": "anthropic/claude-3.5-sonnet", "api_key": os.getenv("OPENROUTER_API_KEY", "")},
        "openai":     {"base_url": "https://api.openai.com/v1", "model": "gpt-4o-mini",           "api_key": os.getenv("OPENAI_API_KEY", "")},
    }
    cfg = PROVIDERS[PROVIDER]
    client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])

    review_prompt = '''Streamlit 장르 예측 앱에 신뢰도 bar chart와 st.session_state 이력을 추가했습니다.
    predict_with_confidence()는 RandomForest.predict_proba()의 10개 확률을 정렬해 반환합니다.
    1) 왜 1위 장르만 보여주는 것보다 확률 분포 전체를 보여주는 게 사용자에게 더 정직한지,
    2) st.session_state가 왜 필요한지(Streamlit의 재실행 모델과 연결해서),
    3) freeze_backbone=True와 False 중 GTZAN처럼 데이터가 940장뿐인 상황엔 어느 쪽이 더 안전한지
    각각 근거를 들어 설명해줘.'''

    try:
        resp = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": review_prompt}],
        )
        print(resp.choices[0].message.content)
    except Exception as e:
        print(f'[안내] {PROVIDER} 서버/키 연결 실패({e}) — 서버를 켜거나 PROVIDER를 바꿔 재시도하세요.')

[안내] lmstudio 서버/키 연결 실패(Connection error.) — 서버를 켜거나 PROVIDER를 바꿔 재시도하세요.


### 3️⃣ Debug — "조용히 틀린" 코드 찾기
아래는 **실행은 되고 에러도 안 나지만, 조용히 잘못된 확률**을 내는 코드입니다. 결함을 찾아 고치세요.

In [49]:
# [Debug] 아래 코드는 에러 없이 실행됩니다 — 그런데 조용히 잘못된 확률을 냅니다. 결함을 찾아 고쳐보세요.
def predict_with_confidence_bug(wav_path: str) -> tuple:
    vec = extract_features(wav_path)      # (1, 57)
    # ⚠️ 여기를 의심해보세요 — sc.transform(vec)을 빼먹었습니다
    proba = rf.predict_proba(vec)[0]
    prob_dict = {le.classes_[i]: float(proba[i]) for i in range(len(proba))}
    prob_dict = dict(sorted(prob_dict.items(), key=lambda x: x[1], reverse=True))
    return list(prob_dict.keys())[0], prob_dict

if LIBROSA_AVAILABLE and test_files:
    _, buggy_probs = predict_with_confidence_bug(test_files[0][1])
    _, normal_probs = predict_with_confidence(test_files[0][1])
    print('버그 버전 1위 확률 :', list(buggy_probs.values())[0])
    print('정상 버전 1위 확률 :', list(normal_probs.values())[0])
    print('두 결과가 다르다면 스케일링을 빼먹은 영향입니다.')

버그 버전 1위 확률 : 0.27
정상 버전 1위 확률 : 0.82
두 결과가 다르다면 스케일링을 빼먹은 영향입니다.


💭 **생각해 볼 점**:
- `rf.fit()`은 `sc.transform()`을 거친 스케일된 벡터로 학습했습니다. 추론할 때 스케일링을 빼먹으면 RandomForest는 왜 에러 없이 그냥 (틀린) 확률을 내놓을까요? (트리 기반 모델은 피처의 절대 스케일에 상대적으로 둔감하지만, 각 트리의 분기 임계값은 학습 시점의 스케일된 값 기준으로 정해져 있습니다 — 스케일이 다른 입력이 들어오면 엉뚱한 분기를 타게 됩니다.)
- 이 버그는 shape 에러가 안 납니다 — extract_features의 출력 (1, 57)이 predict_proba가 기대하는 shape과 정확히 같기 때문입니다. **shape이 맞는다고 로직이 맞는 건 아닙니다.**

### 4️⃣ Prompt Card
📝 **카드 1~3** (이 단원 실험 설계형):
```text
1. "GTZAN 10장르 중 어느 두 장르 쌍이 가장 자주 서로 헷갈리는지(2위 확률이 가장 높은 경우) 5곡씩 뽑아 확인하는 코드를 같이 짜줘."
2. "freeze_backbone=True/False 두 모델의 trainable parameter 수 차이를 표로 만들고, 데이터 940장 기준으로 어느 쪽이 과적합 위험이 낮을지 근거를 들어 설명해줘."
3. "confidence_score(표준편차) 대신 엔트로피(entropy)로 확신도를 계산하면 결과가 어떻게 달라질지 같이 코드로 확인해줘."
```
🎯 **마무리 체크**: [ ] Solo 3레벨 직접 [ ] Review 실행검증 [ ] Debug 결함 찾음 [ ] Prompt Card 1개 내 노트에

---
## 📚 [세션 요약] Streamlit 앱 고급 + ResNet-18 도입

| 개념 | 핵심 한 줄 |
|------|------------|
| `predict_with_confidence()` | 10장르 전체 확률 반환 — 점 예측보다 정직 |
| 신뢰도 bar chart | 확률 분포 시각화 — 50% 미만이면 불확실 경고 |
| `st.session_state` | Streamlit의 메모리 — 재실행 시에도 값 유지 |
| `accept_multiple_files=True` | 멀티파일 업로드 — grouped bar chart 비교 |
| ResNet-18 전이학습 | FC(512→1000) → FC(512→10) 교체만으로 도메인 변환 |
| `freeze_backbone` | feature extractor 모드 — 데이터 적을 때 유리 |

> 🎯 **핵심 Takeaways**
> 1. **이론적 근거**: 점 예측(1위만)보다 확률 분포 전체를 보여주는 것이 모델의 실제 확신 수준을 정직하게 전달한다.
> 2. **실무적 활용**: 전이학습은 "이미 학습된 필터를 재사용하고 마지막 레이어만 새로 학습"하는 것 — 데이터가 적을 때 CNN을 쓸 수 있게 해주는 핵심 전략이다.
>
> ➡️ **다음 단계**: `8강_ResNet18_스펙트로그램분류(AI_Pair).ipynb` — 오늘 구조만 이해한 ResNet-18을 실제로 fine-tuning해 RF 90% 대비 정확도가 얼마나 오르는지 직접 확인합니다.

In [50]:
# ┌────────────────────────────────────────────────────────┐
# │ [최종 셀 16/16] 7강 실습 완료 체크리스트               │
# └────────────────────────────────────────────────────────┘
print('=' * 58)
print('  7강 실습 완료 체크리스트')
print('=' * 58)

checks = [
    ('6강 모델 로드 (또는 즉석 학습)',           MODEL_PATH.exists()),
    ('predict_with_confidence() 정의',           True),
    ('신뢰도 bar chart 시각화',                  True),
    ('session_state 이력 시뮬레이션',            len(prediction_history) > 0),
    ('멀티파일 grouped bar chart',              True),
    ('ResNet-18 아키텍처 다이어그램',            True),
    ('load_resnet_model() 정의 + 파라미터 확인', True),
    ('★★ PNG 단일 추론',                       RUN_HEAVY),
    ('app_v2.py 저장',                         (NB_DIR / 'app_v2.py').exists()),
]

done = 0
for name, ok in checks:
    status = 'DONE' if ok else ('SKIP(GPU)' if 'PNG' in name else 'MISS')
    print(f'  [{status:10s}] {name}')
    if ok:
        done += 1

print()
print(f'  완료: {done}/{len(checks)}')
print()
print('  다음 수업: 8강 — ResNet-18 Fine-tuning + RF 성능 비교')
print('=' * 58)

  7강 실습 완료 체크리스트
  [DONE      ] 6강 모델 로드 (또는 즉석 학습)
  [DONE      ] predict_with_confidence() 정의
  [DONE      ] 신뢰도 bar chart 시각화
  [DONE      ] session_state 이력 시뮬레이션
  [DONE      ] 멀티파일 grouped bar chart
  [DONE      ] ResNet-18 아키텍처 다이어그램
  [DONE      ] load_resnet_model() 정의 + 파라미터 확인
  [DONE      ] ★★ PNG 단일 추론
  [DONE      ] app_v2.py 저장

  완료: 9/9

  다음 수업: 8강 — ResNet-18 Fine-tuning + RF 성능 비교
